# SMAQ Triton Kernel Validation

This notebook is self-contained. It includes an inline copy of the current SMAQ scalar quantizer and Triton `dequant + score` kernel so it can run directly on Google Colab.

Recommended runtime:
- GPU enabled
- T4 is enough for correctness and first timing checks

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(module_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or module_name])

ensure_package('torch', 'torch')
ensure_package('triton', 'triton')
ensure_package('pandas', 'pandas')

import math
import time
from typing import NamedTuple

import pandas as pd
import torch
import torch.nn.functional as F
import triton
import triton.language as tl
from IPython.display import display

print('torch', torch.__version__)
print('triton', triton.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('device', props.name)
    print('compute capability', f"{props.major}.{props.minor}")
    print('total memory (GB)', round(props.total_memory / (1024**3), 2))

In [ ]:
def ssf_log(eigvals: torch.Tensor, c: float = 5.0) -> torch.Tensor:
    shaped = torch.log1p(c * eigvals.clamp(min=0))
    log_shaped = torch.log(shaped.clamp(min=1e-8))
    log_shaped = log_shaped - log_shaped.mean()
    return torch.exp(log_shaped)


def build_smaq_metric(Sigma_q: torch.Tensor, c: float = 5.0) -> tuple[torch.Tensor, torch.Tensor]:
    evals, evecs = torch.linalg.eigh(Sigma_q)
    shaped_evals = ssf_log(evals, c)
    sqrt_diag = torch.diag(shaped_evals.sqrt())
    inv_sqrt_diag = torch.diag(1.0 / shaped_evals.sqrt())
    E = evecs @ sqrt_diag @ evecs.T
    E_inv = evecs @ inv_sqrt_diag @ evecs.T
    return E, E_inv


class SMAQQuantized(NamedTuple):
    indices: torch.Tensor
    norms: torch.Tensor
    bits: int


def _pack_indices(indices: torch.Tensor, bits: int) -> torch.Tensor:
    vals_per_byte = max(1, 8 // bits)
    batch_shape = indices.shape[:-1]
    d = indices.shape[-1]

    padded_d = ((d + vals_per_byte - 1) // vals_per_byte) * vals_per_byte
    if padded_d > d:
        indices = F.pad(indices.to(torch.uint8), (0, padded_d - d), value=0)

    reshaped = indices.to(torch.uint8).reshape(*batch_shape, -1, vals_per_byte)
    shifts = torch.arange(vals_per_byte, device=indices.device, dtype=torch.uint8) * bits
    return (reshaped << shifts).sum(dim=-1, dtype=torch.uint8)


def _unpack_indices(packed: torch.Tensor, bits: int, d: int) -> torch.Tensor:
    vals_per_byte = max(1, 8 // bits)
    mask = (1 << bits) - 1
    shifts = torch.arange(vals_per_byte, device=packed.device, dtype=torch.uint8) * bits
    unpacked = ((packed.unsqueeze(-1) >> shifts) & mask).reshape(*packed.shape[:-1], -1)
    return unpacked[..., :d].long()


def _normal_ppf(probs: torch.Tensor) -> torch.Tensor:
    return math.sqrt(2.0) * torch.erfinv((2.0 * probs) - 1.0)


class SMAQQuantizer(torch.nn.Module):
    def __init__(
        self,
        dim: int,
        Sigma_q: torch.Tensor | None = None,
        bits: int = 3,
        device: torch.device | None = None,
        dtype: torch.dtype = torch.float32,
        E: torch.Tensor | None = None,
        E_inv: torch.Tensor | None = None,
    ):
        super().__init__()
        self.dim = dim
        self.bits = bits
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        metric_dtype = dtype

        if E is None or E_inv is None:
            if Sigma_q is None:
                Sigma_q = torch.eye(dim, dtype=torch.float32, device=self.device)
            E, E_inv = build_smaq_metric(Sigma_q.to(torch.float32), c=5.0)

        self.register_buffer('E', E.to(self.device, dtype=metric_dtype))
        self.register_buffer('E_inv', E_inv.to(self.device, dtype=metric_dtype))

        probs = torch.linspace(0.0, 1.0, (2**bits) + 2, device=self.device, dtype=torch.float32)[1:-1]
        centroids = _normal_ppf(probs).to(metric_dtype)

        boundaries_p = torch.linspace(0.0, 1.0, (2**bits) + 1, device=self.device, dtype=torch.float32)
        clipped = boundaries_p.clamp(1e-6, 1.0 - 1e-6)
        boundaries = _normal_ppf(clipped).to(metric_dtype)
        boundaries[0] = -float('inf')
        boundaries[-1] = float('inf')

        self.register_buffer('centroids', centroids)
        self.register_buffer('boundaries', boundaries)
        self.register_buffer('decision_boundaries', boundaries[1:-1].contiguous())

    def rotate_query(self, query: torch.Tensor) -> torch.Tensor:
        return torch.matmul(query.float(), self.E_inv.T)

    def quantize(self, k: torch.Tensor) -> SMAQQuantized:
        norms = k.norm(dim=-1, keepdim=False)
        k_unit = k / (norms.unsqueeze(-1) + 1e-10)
        y = torch.matmul(k_unit.float(), self.E.T)
        indices = torch.searchsorted(self.decision_boundaries, y.contiguous())
        packed = _pack_indices(indices, self.bits)
        return SMAQQuantized(indices=packed, norms=norms, bits=self.bits)

    def attention_score_ref(self, query: torch.Tensor, quantized_key: SMAQQuantized) -> torch.Tensor:
        query_rot = self.rotate_query(query)
        indices = _unpack_indices(quantized_key.indices, quantized_key.bits, self.dim)
        y_hat = self.centroids[indices]
        scores = torch.matmul(query_rot.float(), y_hat.float().transpose(-2, -1))
        return scores * quantized_key.norms.unsqueeze(-2)


In [ ]:
@triton.jit
def _smaq_scores_kernel(
    query_ptr,
    packed_ptr,
    centroids_ptr,
    norms_ptr,
    out_ptr,
    n_qh,
    n_kv,
    packed_dim,
    d_model,
    query_stride_h,
    query_stride_d,
    packed_stride_t,
    packed_stride_p,
    norm_stride_t,
    out_stride_h,
    out_stride_t,
    bits: tl.constexpr,
    vals_per_byte: tl.constexpr,
    block_q: tl.constexpr,
    block_t: tl.constexpr,
):
    pid_q = tl.program_id(0)
    pid_t = tl.program_id(1)

    q_offsets = pid_q * block_q + tl.arange(0, block_q)
    t_offsets = pid_t * block_t + tl.arange(0, block_t)

    q_mask = q_offsets < n_qh
    t_mask = t_offsets < n_kv

    acc = tl.zeros((block_q, block_t), dtype=tl.float32)
    index_mask = (1 << bits) - 1

    for p in range(0, packed_dim):
        packed_vals = tl.load(
            packed_ptr + (t_offsets * packed_stride_t) + (p * packed_stride_p),
            mask=t_mask,
            other=0,
        ).to(tl.int32)

        for v in range(0, vals_per_byte):
            d_idx = p * vals_per_byte + v
            if d_idx < d_model:
                indices = (packed_vals >> (v * bits)) & index_mask
                y_hat = tl.load(centroids_ptr + indices, mask=t_mask, other=0.0)
                q_vec = tl.load(
                    query_ptr + (q_offsets * query_stride_h) + (d_idx * query_stride_d),
                    mask=q_mask,
                    other=0.0,
                ).to(tl.float32)
                acc += q_vec[:, None] * y_hat[None, :]

    norms = tl.load(norms_ptr + t_offsets * norm_stride_t, mask=t_mask, other=0.0).to(tl.float32)
    acc *= norms[None, :]

    tl.store(
        out_ptr + (q_offsets[:, None] * out_stride_h) + (t_offsets[None, :] * out_stride_t),
        acc,
        mask=q_mask[:, None] & t_mask[None, :],
    )


def triton_attention_scores(quantizer, query: torch.Tensor, quantized_key: SMAQQuantized) -> torch.Tensor:
    query_rot = quantizer.rotate_query(query).contiguous().to(torch.float32)
    packed = quantized_key.indices.contiguous()
    norms = quantized_key.norms.contiguous().to(torch.float32)
    centroids = quantizer.centroids.contiguous().to(torch.float32)

    n_qh, d_model = query_rot.shape
    n_kv, packed_dim = packed.shape
    vals_per_byte = max(1, 8 // int(quantized_key.bits))

    out = torch.empty((n_qh, n_kv), device=query.device, dtype=torch.float32)
    block_q = 8
    block_t = 64

    grid = (triton.cdiv(n_qh, block_q), triton.cdiv(n_kv, block_t))

    _smaq_scores_kernel[grid](
        query_rot,
        packed,
        centroids,
        norms,
        out,
        n_qh,
        n_kv,
        packed_dim,
        d_model,
        query_rot.stride(0),
        query_rot.stride(1),
        packed.stride(0),
        packed.stride(1),
        norms.stride(0),
        out.stride(0),
        out.stride(1),
        bits=int(quantized_key.bits),
        vals_per_byte=vals_per_byte,
        block_q=block_q,
        block_t=block_t,
        num_warps=4,
    )
    return out


In [ ]:
assert torch.cuda.is_available(), 'Please switch Colab to a GPU runtime.'

device = torch.device('cuda')
dim = 128
bits = 3
num_query_heads = 8
num_tokens = 1024

torch.manual_seed(0)
Sigma_q = torch.randn(dim, dim, device=device)
Sigma_q = Sigma_q @ Sigma_q.T + 1e-3 * torch.eye(dim, device=device)

quantizer = SMAQQuantizer(dim=dim, Sigma_q=Sigma_q, bits=bits, device=device)
keys = torch.randn(num_tokens, dim, device=device)
query = torch.randn(num_query_heads, dim, device=device)

qk = quantizer.quantize(keys)
print('packed indices shape', tuple(qk.indices.shape))
print('norms shape', tuple(qk.norms.shape))


In [ ]:
scores_ref = quantizer.attention_score_ref(query, qk)
scores_tri = triton_attention_scores(quantizer, query, qk)

max_abs = (scores_ref - scores_tri).abs().max().item()
mean_abs = (scores_ref - scores_tri).abs().mean().item()

print('max abs diff', max_abs)
print('mean abs diff', mean_abs)
print('allclose', torch.allclose(scores_ref, scores_tri, atol=1e-4, rtol=1e-4))

In [ ]:
def bench(fn, warmup=20, iters=100):
    for _ in range(warmup):
        out = fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        out = fn()
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) * 1000.0 / iters

torch_ms = bench(lambda: quantizer.attention_score_ref(query, qk))
triton_ms = bench(lambda: triton_attention_scores(quantizer, query, qk))

summary_df = pd.DataFrame([
    {
        'gpu': torch.cuda.get_device_name(0),
        'num_tokens': num_tokens,
        'head_dim': dim,
        'num_query_heads': num_query_heads,
        'bits': bits,
        'torch_ms': round(torch_ms, 4),
        'triton_ms': round(triton_ms, 4),
        'speedup_x': round(torch_ms / triton_ms, 3) if triton_ms > 0 else None,
    }
])
display(summary_df)


In [ ]:
# Optional sweep for a few token lengths.
rows = []
for num_tokens in [256, 512, 1024, 2048, 4096]:
    keys = torch.randn(num_tokens, dim, device=device)
    qk = quantizer.quantize(keys)
    torch_ms = bench(lambda: quantizer.attention_score_ref(query, qk), warmup=10, iters=50)
    triton_ms = bench(lambda: triton_attention_scores(quantizer, query, qk), warmup=10, iters=50)
    rows.append({
        'num_tokens': num_tokens,
        'torch_ms': round(torch_ms, 4),
        'triton_ms': round(triton_ms, 4),
        'speedup_x': round(torch_ms / triton_ms, 3) if triton_ms > 0 else None,
    })

sweep_df = pd.DataFrame(rows)
display(sweep_df)